In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
# eval_interval = 2500
learning_rate = 3e-3
eval_iters = 250

cuda


In [2]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


In [3]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
# print(data[:100])

In [4]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]
#Generate random training examples
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y = x.to(device), y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

inputs:
tensor([[68, 59,  1, 74, 67, 58, 54, 72],
        [ 1, 54, 65, 65,  1, 58, 65, 72],
        [ 1, 73, 61, 58,  1, 40, 71, 62],
        [57, 78,  1, 71, 74, 67, 67, 62]], device='cuda:0')
targets:
tensor([[59,  1, 74, 67, 58, 54, 72, 62],
        [54, 65, 65,  1, 58, 65, 72, 58],
        [73, 61, 58,  1, 40, 71, 62, 67],
        [78,  1, 71, 74, 67, 67, 62, 67]], device='cuda:0')


In [5]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [6]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


Dj
Kf,5(tNt;C6tnKMZSlKI)0ZRd([SR!&unX )(5C]&wF[!QtRsb-yWXoT0eTJpEVlqC]YNPPnKN9FSBX?F0ebwNISRr5ck
u﻿i98K:;'jWO﻿g:9KV&0*(1Wk5u
Wwk.V&(o5E2(wk3k
&bQyZFFPYN w8FKMWZOXNx(S
kwmkF[UVuoxpO
a[y'h.sMiZ)Gi"qfA1CxY;jA:NM'-e.HrzkCXAB﻿z0L﻿oy1&tFIL I9Tdz8FMJ﻿p0﻿VS2]EaJ0S(MtFP-JMAq']uQS9Y(ss9﻿t6.0h)A*7qf!Wb]us1OdhlJpD2]Y3picnK.IVbP
3;AGoAWxR7zO
CJsb-A]ywqUHv"p,!WU2Sh26iLk EuB,,t'[DB&s5ea[sP:1:jP
"NnKKqg:ej?_k06? wGKqPuDOS(0-7SG]u9.*rn(O*nscrN4X.1hg*e8stWig::!f!y﻿g0LtY:wStjV5M.FXCCP:﻿W8&_3n2XA"N
)SAb-]W[hu9Ip(fP


In [7]:
#create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")
    # sample a batch of data
    xb, yb = get_batch('train')
    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.9323, val loss: 4.8921
step: 250, train loss: 4.3494, val loss: 4.3187
step: 500, train loss: 3.8676, val loss: 3.8663
step: 750, train loss: 3.5065, val loss: 3.5155
3.075160264968872
